# Train a sparse dictionary-learning model on UNI patch embeddings

Counterpart of `train_SAE.ipynb` using `MiniBatchDictionaryCoding` (L1 dictionary learning,
PyTorch FISTA backend on GPU).

- `n_components=256`; `alpha` tuned so the mean number of active features per patch is ~20,
  matching SAE1 (65,536 features; mean L0 = 20.5 over the 2,007,406 patches of the 40 slides,
  computed from `tiles_table.X`).
- Nonnegative codes (like SAE ReLU activations), signed atoms, no mean subtraction.
- `scale=True`: inputs are multiplied by one constant so that mean ||x||^2 = 1 (unit norm on
  average). The SAE's scaling gives mean ||x||^2 = sqrt(d) instead, so on this scale the
  SAE's `max_score_threshold=0.1` corresponds to 0.1 / d**0.25 ~= 0.0177.

In [ ]:
from fmsae_repro import MODELS, RESULTS, load_manifest
import ezslide
import lazyslide as zs
import mesoslide as ms
import numpy as np
import pandas as pd

In [ ]:
import os, resource, socket, threading, time
import torch

def _rss_gb():
    """Current resident memory of this process (GB)."""
    with open('/proc/self/status') as f:
        for line in f:
            if line.startswith('VmRSS:'):
                return int(line.split()[1]) / 2**20
    return float('nan')

resource_log = []

class track:
    """Record wall time, peak host RSS (sampled every 0.5 s), process maxrss and peak GPU memory of a block."""
    def __init__(self, step):
        self.step = step

    def __enter__(self):
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        self.rss_start = self.rss_peak = _rss_gb()
        self._stop = threading.Event()
        def sample():
            while not self._stop.wait(0.5):
                self.rss_peak = max(self.rss_peak, _rss_gb())
        self._thread = threading.Thread(target=sample, daemon=True)
        self._thread.start()
        self.t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        torch.cuda.synchronize()
        wall = time.perf_counter() - self.t0
        self._stop.set()
        self._thread.join()
        resource_log.append(dict(
            step=self.step,
            wall_time_s=round(wall, 1),
            rss_start_gb=round(self.rss_start, 2),
            rss_peak_gb=round(max(self.rss_peak, _rss_gb()), 2),
            process_maxrss_gb=round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20, 2),
            gpu_peak_gb=round(torch.cuda.max_memory_allocated() / 2**30, 2),
        ))
        print(resource_log[-1])

print("node:", socket.gethostname(), "| GPU:", torch.cuda.get_device_name(0),
      "| CPU cores available:", len(os.sched_getaffinity(0)))

In [ ]:
manifest = load_manifest()
with track(f"open slides ({len(manifest)})"):
    slides = ezslide.open_slides(manifest, attach_images=False)

## Tune `alpha` to ~20 active features per patch
One epoch per `alpha` on a 100k-patch subsample, then log-log interpolation.

In [ ]:
import joblib
from mesoslide.tools import MiniBatchDictionaryCoding

out_dir = f'{MODELS}/DL'  # checkpoints: one model per alpha, plus the full fit
os.makedirs(out_dir, exist_ok=True)

with track("stack UNI embeddings"):
    X_all = np.vstack([slide['tiles_table'].obsm['UNI_embedding'] for _, slide in slides.items()])
print(X_all.shape, X_all.dtype)

dl_params = dict(n_components=256, scale=True, center=False,
                 positive_code=True, positive_dict=False,
                 batch_size=2048, device='cuda', random_state=0,
                 dict_kwargs={'tol': 0.0, 'max_no_improvement': None})  # no early stopping: full epochs

rng = np.random.default_rng(0)
X_sub = X_all[rng.choice(len(X_all), size=100_000, replace=False)]

target_l0 = 20
alphas = [0.003, 0.01, 0.03, 0.1, 0.3]
l0s = []
for a in alphas:
    with track(f"alpha scan: alpha={a}, 100k patches, 1 epoch"):
        m = MiniBatchDictionaryCoding(alpha=a, max_iter=1, **dl_params).fit(X_sub)
    joblib.dump(m, f"{out_dir}/scan_alpha_{a}.joblib")
    l0s.append(m._training_log['mean_l0'])
    print(f"alpha={a:<5} mean L0={l0s[-1]:7.1f}")

# Mean L0 decreases with alpha; interpolate log(alpha) against log(L0).
log_l0, log_a = np.log(l0s)[::-1], np.log(alphas)[::-1]
if not log_l0[0] <= np.log(target_l0) <= log_l0[-1]:
    print("WARNING: target L0 outside the scanned range; extend `alphas`.")
alpha = float(np.exp(np.interp(np.log(target_l0), log_l0, log_a)))
print(f"chosen alpha = {alpha:.4f}")

## Fit on all patches (2000 steps x batch 2048 as for the SAE; early stopping off)

In [ ]:
# 2000 steps x batch 2048, the same schedule as the SAE (num_steps=2000, batch_size=2048).
dl = MiniBatchDictionaryCoding(alpha=alpha, max_steps=2000, **dl_params)
with track(f"full fit: alpha={alpha:.4f}, {len(X_all):,} patches, 2000 steps x 2048"):
    dl.fit(X_all)
joblib.dump(dl, f"{out_dir}/dl_model.joblib")
print("scale_factor_:", dl.scale_factor_)
print(dl._training_log)

Reconstruction on a random 100k-patch sample, in the model's scaled input space (mean ||x||^2 = d).
FVU = sum ||x - x_hat||^2 / sum ||x - mean(x)||^2.

In [ ]:
X_eval = X_all[rng.choice(len(X_all), size=100_000, replace=False)].astype(np.float64)
C_eval = dl.transform(X_eval)
Xs = X_eval * dl.scale_factor_
Xs_hat = np.asarray(C_eval @ dl.components_.astype(np.float64)) + dl.mean_
sse = ((Xs - Xs_hat) ** 2).sum()
print(f"MSE per coordinate: {sse / Xs.size:.4f}")
print(f"FVU (vs per-dimension mean): {sse / ((Xs - Xs.mean(0)) ** 2).sum():.4f}")
print(f"FVU (vs zero): {sse / (Xs ** 2).sum():.4f}")
del X_all, X_sub, X_eval, C_eval, Xs, Xs_hat

In [ ]:
# Whole-slide encoding via the model's obsm_key path. feature_extraction(sparse_transform=...)
# calls the transform on 32-row batches, which is slow for an iterative (FISTA) encoder.
with track(f"transform all patches ({len(slides)} slides)"):
    dl.transform(list(slides.values()), obsm_key="UNI_embedding", tile_key="tiles",
                 sparse_key_added="UNI_DL_v1", save=False)

# Actual L0 over all patches (active features per patch).
l0 = np.concatenate([
    np.diff(s['tiles_table'][:, [f'UNI_DL_v1_{i}' for i in range(dl.n_components)]].X.tocsr().indptr)
    for _, s in slides.items()])
print(f"L0 over {len(l0):,} patches: mean {l0.mean():.2f}, median {np.median(l0):.0f}, "
      f"p10/p90 {np.percentile(l0, 10):.0f}/{np.percentile(l0, 90):.0f}")

## Feature selection
`pct_threshold=0.002` as for the SAE. `max_score_threshold=0.0177` is the SAE-equivalent value on
this input scale (0.1 / 1024**0.25 ~= 0.0177). The distribution below is one continuous cloud with no
low-score tail (same in the 4-slide pilot), so there is no natural cutoff and all features are kept.

In [ ]:
from mesoslide.tools import FeatureSelector

selector = FeatureSelector(
    pct_threshold=0.002,
    max_score_threshold=0.0177,
    n_chunks=40
)
selector.compute_activation_stats(slides,
                                  feature_prefix='UNI_DL_v1',
                                  num_features=256)
g = selector.plot_feature_selection()

In [ ]:
highly_active_features_idx = selector.get_selected_indices()
len(highly_active_features_idx)

In [ ]:
# Cluster
from mesoslide.tools import FeatureClusterer

n_clusters = min(25, len(highly_active_features_idx))
clusterer = FeatureClusterer(high_activation_threshold=0.5)
clusterer.compute_iou(slides, feature_prefix='UNI_DL_v1',
                      feature_indices=highly_active_features_idx)
clusterer.cluster(threshold=n_clusters, criterion='maxclust')

In [ ]:
g = clusterer.plot_heatmap()

In [ ]:
exemplar_patches = ms.select_exemplar_patches(slides, [f'UNI_DL_v1_{idx}' for idx in highly_active_features_idx])
exemplar_patches

In [ ]:
from mesoslide.plotting._patch_gallery import plot_feature_gallery
from mesoslide.preprocessing import extract_patch_images
import distinctipy
from matplotlib.colors import ListedColormap

# ── configure ──────────────────────────────────────────────────────────────────
groups_to_plot = list(range(1, n_clusters + 1))
colors = distinctipy.get_colors(n_clusters, rng=43)
cmap = ListedColormap(colors)

# ── extract pixel patches once, keyed by global feature index ──────────────────
exemplar_images = extract_patch_images(exemplar_patches, slides, channel_first=False, progress_bar=True)
feature_to_image = {
    int(name.rsplit('_', 1)[-1]): img
    for name, img in zip(exemplar_patches.obs['_feature_name'], exemplar_images)
}

# ── filter by group ────────────────────────────────────────────────────────────
ordered_idx      = clusterer.feature_indices_[clusterer.reordered_idx_]
reordered_groups = clusterer.reordered_clusters_.tolist()

mask      = [i for i, g in enumerate(reordered_groups) if g in groups_to_plot]
patches   = [feature_to_image[ordered_idx[i]] for i in mask]
group_ids = [reordered_groups[i] for i in mask]
labels    = [str(ordered_idx[i]) for i in mask]

# ── plot ───────────────────────────────────────────────────────────────────────
fig, axs = plot_feature_gallery(
    images=patches,
    group_ids=group_ids,
    labels=labels,
    n_cols=10,
    patch_size=1.0,
    border_extend=0.05,
    border_alpha=1.0,
    cmap=cmap,
    fontsize=6,
)

## Top patches for two selected features

In [ ]:
sel = highly_active_features_idx
most_frequent = int(sel[np.argmax(selector.pct_active_[sel])])
highest_peak = int(sel[np.argmax(selector.max_score_[sel])])
print(f"most frequently active: UNI_DL_v1_{most_frequent} "
      f"(active in {selector.pct_active_[most_frequent]:.1%} of patches)")
print(f"highest peak score:     UNI_DL_v1_{highest_peak} "
      f"(max score {selector.max_score_[highest_peak]:.2f})")

In [ ]:
patches = ms.select_top_patches(slides, f'UNI_DL_v1_{most_frequent}', 20, top_fraction=0.1)
ms.plotting.plot_patch_gallery(patches, slides, return_fig=True)

In [ ]:
patches = ms.select_top_patches(slides, f'UNI_DL_v1_{highest_peak}', 20, top_fraction=0.1)
ms.plotting.plot_patch_gallery(patches, slides, return_fig=True)

In [ ]:
fig, ax = ms.plotting.plot_feature_map(slides['LSP24530'], f'UNI_DL_v1_{most_frequent}',
                                       figsize=(10, 10), fill_alpha=1.0, return_fig=True)
fig

In [ ]:
fig, ax = ms.plotting.plot_feature_map(slides['LSP24530'], f'UNI_DL_v1_{highest_peak}',
                                       figsize=(10, 10), fill_alpha=1.0, return_fig=True)
fig

## Running time and memory

In [ ]:
usage = pd.DataFrame(resource_log)
usage.to_csv(f"{RESULTS}/tables/DL_resource_usage.csv", index=False)
usage

In [ ]:
joblib.dump(dl, f"{out_dir}/dl_model.joblib")